In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

figure_dir = "figures/supp-figures"
setup_plotting(figure_dir, display_dpi=200, save_dpi=300)

import pandas as pd
import scanpy as sc

## Load data

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/08.2-kidney_tcr_clonal_clusters_peri_glom_annot.h5ad"
adata = sc.read_h5ad(path)

# remove control samples
adata = adata[adata.obs["condition"] == "ANCA"].copy()
sample_mapping = {k: f"S{i}" for i, k in enumerate(adata.obs["sample"].unique())}
adata.obs["sample_short"] = adata.obs["sample"].map(sample_mapping)
adata

In [ ]:
adata.obs.columns

In [ ]:
adata.obs["avbv_cluster_filtered"]

In [ ]:
# get cd8 dominant ccs
ad_t = adata[adata.obs["cell_type_l1"] == "T"].copy()
ad_cluster = adata[adata.obs["avbv_cluster"].notna()].copy()
ct_key = "cell_type_l1.1"
df = (
    ad_cluster.obs.groupby("avbv_cluster", observed=True)[ct_key]
    .value_counts(normalize=True)
    .unstack()
)
cd8_dominant_ccs = df.index[df["CD8+"] >= 0.5].tolist()
adata.obs["cd8_dominant_ccs"] = adata.obs["avbv_cluster"].astype(str)
adata.obs.loc[
    ~adata.obs["cd8_dominant_ccs"].isin(cd8_dominant_ccs), "cd8_dominant_ccs"
] = pd.NA
print(f"Number of CD8+ ccs: {len(cd8_dominant_ccs)}")

## Check localization of CD8+ clusters in periglomerular and tubulointerstitial regions

In [ ]:
adata.obs["in_tubuloint"] = (~adata.obs["in_glom"]) & (~adata.obs["in_peri_glom"])
adata.obs["region"] = "Tubulointerstitial"
adata.obs.loc[adata.obs["in_glom"], "region"] = "Glomerular"
adata.obs.loc[adata.obs["in_peri_glom"], "region"] = "Periglomerular"

sc.pl.spatial(
    adata,
    color=["cd8_dominant_ccs", "region"],
    title=["CD8+ dominant ccs", "Region"],
    spot_size=10,
)

## Compare CD8+ clonal clusters vs CD8+ T cells per region


In [ ]:
from spatial_tcr.plotting import plot_cd8_clonal_clusters_vs_cells_stacked

plot_cd8_clonal_clusters_vs_cells_stacked(
    adata,
    region_col="region",
    cc_col="cd8_dominant_ccs",
    cell_type_col="cell_type_l1.1",
    cell_type="CD8+",
    exclude_regions=["Glomerular"],
    figsize=(2, 5),
    show_absolute_counts=True,
    test_method="chi2",
    annotate_significance=False,
)

In [ ]:
ncols = 5
plot_cd8_clonal_clusters_vs_cells_stacked(
    adata,
    region_col="region",
    cc_col="cd8_dominant_ccs",
    cell_type_col="cell_type_l1.1",
    cell_type="CD8+",
    exclude_regions=["Glomerular"],
    figsize=(ncols * 0.75, 8),
    ncols=ncols,
    sample_col="sample_short",
    show_raw_pvalues=True,
    show_absolute_counts=True,
    test_method="fisher",
)

In [ ]:
from spatial_tcr.plotting import plot_cd8_clonal_clusters_vs_cells_per_region

plot_cd8_clonal_clusters_vs_cells_per_region(
    adata,
    normalize=True,
    title="CD8+ clonal clusters vs CD8+ T cells per region",
    exclude_regions=["Glomerular"],
    figsize=(4, 6),
)